# Azure BioSystems Cielo 6

The Azure BioSystems Cielo 6 is a six-channel real-time PCR instrument. PyLabRobot communicates directly with its firmware through a USB serial connection.

```{device-card} azure-biosystems-cielo-6
```

The backend supports instrument identity and state, live run progress, workspace and stored-program discovery, thermal protocol execution, and verified `.AZE` result download. Results include raw and processed amplification measurements and, when present, melting-curve measurements.

## Before starting

Connect the Cielo 6 to the computer by USB and identify its serial port. The instrument uses a generic FTDI USB identifier, so you must select the port explicitly. Typical port names are `COM3` on Windows, `/dev/ttyUSB0` on Linux, and `/dev/cu.usbserial-XXXXXXXX` on macOS.

The setup, status, discovery, and result-download sections are read-only. The **Run a protocol** section starts a physical run and heats the block. Interrupting Python or setting a PLR timeout does not stop a physical run; use `stop_run()` when you intend to stop the instrument. `pause_run()` and `resume_run()` change an active run. `delete_program()` permanently deletes a stored program.

## Setup

Create the device, open the serial connection, and verify the identity returned by the firmware.

In [ ]:
from pylabrobot.azure_biosystems import Cielo6

cielo = Cielo6(port="/dev/cu.usbserial-XXXXXXXX")  # replace with your port
await cielo.setup()
cielo.identity

## Read instrument state

Request the current firmware state and temperatures. This request does not change the instrument.

In [ ]:
status = await cielo.request_status()
{
  "state": status.work_state.name,
  "block_temperatures": status.block_temperatures,
  "hot_lid_temperature": status.hot_lid_temperature,
  "progress": status.progress,
}

## Discover stored programs and results

List each workspace and its stored programs. A stored program supplies the instrument-specific optical channels, exposure settings, and lid settings needed to compile a PLR thermal protocol.

In [ ]:
workspaces = await cielo.request_workspace_summary()
workspaces

Choose a stored qPCR program that uses the optical channels and lid settings required by your assay. Retrieving it validates every transport frame and the program CRC32. Replace the names below with values from `workspaces`.

In [ ]:
workspace = "Public"
template_name = "Existing qPCR template"
template = await cielo.request_program(workspace, template_name)
{
  "channels": template.channels,
  "sample_volume": template.sample_volume,
  "step_count": template.step_count,
  "cycle_count": template.cycle_count,
}

Stored result discovery and download are also read-only. `request_experiment_data()` verifies the firmware-provided MD5 before it returns the `.AZE` bytes.

In [ ]:
experiments = await cielo.request_experiment_summary()
experiments[-5:]

In [ ]:
from pylabrobot.azure_biosystems import Cielo6ResultFile

experiment = experiments[-1]
aze_data = await cielo.request_experiment_data(experiment)
stored_result = Cielo6ResultFile.from_bytes(aze_data)
{
  "workspace": stored_result.workspace,
  "program": stored_result.program,
  "raw_amplification_points": len(stored_result.collection_points),
  "processed_amplification_points": len(stored_result.processed_collection_points),
  "melting_points": len(stored_result.melt_records),
}

## Define a qPCR protocol

A `Cielo6ThermalProtocol` contains constant-temperature steps and an optional repeated group. `repeat_from_step` is a zero-based index, and `cycles` includes the first execution of the repeated group. Set `collect_fluorescence=True` on each step that must produce an amplification measurement.

This example performs initial denaturation followed by 40 two-step qPCR cycles. Adjust the temperatures, times, cycle count, and sample volume for your assay.

In [ ]:
from pylabrobot.azure_biosystems import Cielo6ThermalProtocol, Cielo6ThermalStep

protocol = Cielo6ThermalProtocol(
  steps=(
    Cielo6ThermalStep(temperature=95, hold_time=180),
    Cielo6ThermalStep(temperature=95, hold_time=15),
    Cielo6ThermalStep(
      temperature=60,
      hold_time=30,
      collect_fluorescence=True,
    ),
  ),
  repeat_from_step=1,
  cycles=40,
  sample_volume=20,
)

Compile the protocol locally to inspect the exact stored-program model before starting the instrument. Compilation preserves documented device settings from `template` and replaces its thermal steps. It does not communicate with the Cielo.

In [ ]:
program_name = "PLR-qPCR"
compiled = protocol.compile(template, workspace=workspace, name=program_name)
{
  "step_count": compiled.step_count,
  "thermal_step_count": compiled.thermal_step_count,
  "cycle_count": compiled.cycle_count,
  "channels": compiled.channels,
  "sample_volume": compiled.sample_volume,
}

## Run the protocol and monitor progress

The next cell uploads the compiled program, starts a physical run, heats the block, waits for completion, and downloads the verified result. It creates `workspace` if it does not exist. The program transfer is run-scoped and does not add a stored program to the instrument.

Run this cell only after the tubes, consumables, thermal settings, optical channels, and sample volume are correct.

In [ ]:
import asyncio

run_task = asyncio.create_task(
  cielo.run_protocol(
    protocol,
    template=template,
    workspace=workspace,
    program_name=program_name,
    poll_interval=2.0,
  )
)

while not run_task.done():
  state = await cielo.request_run_state()
  print(
    {
      "state": state.status.work_state.name,
      "progress": state.progress,
      "step": state.current_step_index,
      "cycle": state.current_cycle_index,
      "block_temperatures": state.status.block_temperatures,
      "target_temperatures": state.target_temperatures,
      "estimated_completion_at": state.estimated_completion_at,
      "amplification_frames": len(state.amplification_data),
      "melting_frames": len(state.melting_data),
    }
  )
  await asyncio.sleep(2.0)

result = await run_task

To stop an active physical run, call `stop_run()`. Stopping the Python task, interrupting the notebook kernel, or reaching a PLR timeout does not stop the instrument.

In [ ]:
# Run this only when you intend to stop the active run.
# await cielo.stop_run()

## Inspect qPCR results

The completed run returns a `Cielo6ResultFile`. Conversion methods return measurements in PLR plate-data layout: `data[row][column]`, with rows A--H and columns 1--12. Optical channel indices are zero-based.

In [ ]:
raw_amplification = result.to_amplification_results()
processed_amplification = result.to_processed_amplification_results()
melting_curves = result.to_melting_curve_results()

{
  "raw_amplification_measurements": len(raw_amplification),
  "processed_amplification_measurements": len(processed_amplification),
  "melting_curve_measurements": len(melting_curves),
}

For example, collect the processed channel-1 amplification values for well A1. A result can legitimately contain no processed amplification or melting measurements.

In [ ]:
a1_channel_1 = [
  {"cycle": measurement.cycle, "value": measurement.data[0][0]}
  for measurement in processed_amplification
  if measurement.channel_index == 0
]
a1_channel_1

The parsed result can also reproduce the Cielo amplification and melting CSV structures. These methods return text and do not write files.

In [ ]:
amplification_csv = result.to_amplification_csv()
melting_csv = result.to_melting_csv()

## Teardown

Close the PLR serial connection when the workflow is complete. `stop()` disconnects PLR; it does not stop a physical run.

In [ ]:
await cielo.stop()